# Katzen oder Brötchen: Cat or Bun Image Classifier
**Coursework Project**: Computer Vision Image Classification using FastAI & Deep Transfer Learning  
**Inspired by**: Jeremy Howard's FastAI Course (*"Is it a bird?"* & *Fastbook Ch. 1 & 2*)

### Project Overview
The "Cat Loaf vs. Bun" problem is a classic visual ambiguity challenge in modern machine learning: when cats curl up and tuck in their paws, their silhouette, color gradients, and textures bear striking resemblance to baked goods such as bread rolls, croissants, and cinnamon buns.

Rather than treating this as a simplistic binary decision (`is_it_a_cat: True / False`), this notebook designs a **multi-class probability distribution model**. For any input image, it computes normalized confidence percentages for each category (e.g., `84.2% Cat, 15.8% Bun`) using the **Softmax** function, and visually presents the distribution alongside the image.

### Pipeline Outline
1. **Environment & Dependency Setup**: Configuring FastAI, PyTorch, and DuckDuckGo scraping tools.
2. **Resilient Data Acquisition**: Downloading image batches using targeted sub-queries with anti-throttling retry logic.
3. **Data Sanitization**: Detecting and pruning corrupted or unreadable images.
4. **Data Pipeline & Fastbook Presizing**: Constructing `DataBlock` with CPU high-resolution resizing (`Resize(460)`) and GPU batch augmentations (`aug_transforms(224)`).
5. **Transfer Learning with ResNet-34**: Replacing the ImageNet head and fine-tuning with discriminative learning rates and the 1cycle policy.
6. **Error Diagnostics**: Interpreting classification errors via Confusion Matrix and Top Losses inspection.
7. **Model Serialization & Multi-Class Inference**: Exporting `cat_or_bun_model.pkl` and testing with side-by-side probability bar plots.


## 1. Environment Setup & Library Imports

### 1. Practical Goal
Prepare the runtime environment, ensure all essential computer vision and web-scraping packages are available, and initialize hardware acceleration (CUDA GPU if present, CPU fallback).

### 2. Core ML Concept: Reproducibility & Hardware Acceleration
Deep learning workflows require standardized library environments to guarantee numerical reproducibility. While deep neural networks (such as ResNet-34) train significantly faster on specialized parallel hardware (NVIDIA CUDA GPUs), FastAI seamlessly abstracts device management, automatically executing tensor operations on available GPUs or gracefully falling back to CPU cores.

### 3. Python & FastAI Implementation Details
* `!pip install -Uqq`: Shell command option for cloud platforms (Google Colab / Kaggle) to install or upgrade packages quietly.
* `fastai.vision.all`: Imports FastAI's high-level computer vision APIs, PyTorch components, and common utility functions.
* `pathlib.Path`: Provides an object-oriented, cross-platform interface for filesystem path operations.
* `duckduckgo_search.DDGS`: Context-managed client to retrieve search results from DuckDuckGo.


In [ ]:
# Optional: Auto-install dependencies on Google Colab or Kaggle runtimes
# !pip install -Uqq fastai duckduckgo_search

from pathlib import Path
import time
import matplotlib.pyplot as plt
from duckduckgo_search import DDGS
from fastcore.all import *
import fastai
from fastai.vision.all import *

# Verify runtime device (CUDA GPU or CPU)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"FastAI version: {fastai.__version__}")
print(f"PyTorch version: {torch.__version__}")
print(f"Active compute device: {device.upper()}")
if device == 'cuda':
    print(f"GPU device name: {torch.cuda.get_device_name(0)}")


## 2. Resilient Web Image Scraping

### 1. Practical Goal
Define a dependable helper function `search_images()` that queries DuckDuckGo for image URLs, gracefully handling network fluctuations, transient timeouts, and anti-bot rate limits.

### 2. Core ML Concept: Web Scraping Ethics & Rate-Limiting
Modern computer vision often relies on scraping uncurated web images to build custom domain datasets. However, search engines employ automated bot detection and rate limits. If a script issues high-frequency requests without delays, the engine temporarily blocks the IP address (returning HTTP 429 or 202). Robust data ingestion requires courteous scraping: explicit delays, exponential backoff, and retry handling.

### 3. Python & FastAI Implementation Details
* `with DDGS() as ddgs:`: Employs Python's context management protocol to ensure connection pools are cleanly opened and closed.
* `ddgs.images(keywords=term, max_results=max_images)`: Returns a generator yielding dictionaries containing image metadata (`'image'`, `'title'`, `'thumbnail'`).
* `fastcore.all.L`: FastAI's enhanced list structure providing convenient methods like `.itemgot('image')` to extract dictionary keys across all elements.


In [ ]:
def search_images(term, max_images=30, max_retries=3, delay=2):
    """
    Query DuckDuckGo for image URLs with retry logic and rate-limit handling.
    
    Args:
        term (str): Search keywords (e.g., 'cat loaf photo').
        max_images (int): Target number of image URLs to fetch.
        max_retries (int): Retry attempts before giving up.
        delay (int): Base pause duration (seconds) between retry attempts.
        
    Returns:
        fastcore.foundation.L: Extended list containing image URL strings.
    """
    for attempt in range(max_retries):
        try:
            with DDGS() as ddgs:
                results = ddgs.images(keywords=term, max_results=max_images)
                urls = L(results).itemgot('image')
                if len(urls) > 0:
                    print(f"✓ Found {len(urls)} images for '{term}'")
                    return urls
                print(f"⚠ Empty response for '{term}', retrying ({attempt + 1}/{max_retries})...")
        except Exception as e:
            print(f"⚠ Search exception for '{term}' (attempt {attempt + 1}/{max_retries}): {e}")
        time.sleep(delay * (attempt + 1))
    
    print(f"✗ Failed to retrieve images for '{term}' after {max_retries} attempts.")
    return L()


## 3. Targeted Dataset Acquisition

### 1. Practical Goal
Systematically download image batches for both categories (`cat` and `bun`) into structured local subdirectories (`data/cats_and_buns/cat` and `data/cats_and_buns/bun`).

### 2. Core ML Concept: Data Variety & Preventing Shortcut Learning
Neural networks frequently exploit spurious correlations (known as "shortcut learning") if training data lacks environmental variety. For instance, if all cat photos are taken indoors on sofas and all bun photos are on bakery shelves, the network may simply learn background color rather than the subject. We query multiple distinct sub-queries (`'curled up sleeping cat'`, `'cat loaf'`, `'cinnamon roll bun'`, `'bread bun'`) to force the network to focus on morphological shape and texture rather than background context.

### 3. Python & FastAI Implementation Details
* `path / category`: Pathlib operator overloading (`/`) creates platform-independent paths.
* `dest.mkdir(exist_ok=True, parents=True)`: Creates nested directories safely without throwing an error if they already exist.
* `download_images(dest, urls=urls)`: FastAI utility that downloads a list of URLs concurrently using a thread pool.
* `time.sleep(3)`: Polite pause between requests to respect DuckDuckGo service limits.


In [ ]:
# Define root directory for the dataset
path = Path('data/cats_and_buns')

# Targeted query variations to capture morphological ambiguity and environmental variety
category_queries = {
    'cat': [
        'cat photo',
        'curled up sleeping cat',
        'cat loaf photo'
    ],
    'bun': [
        'bread bun photo',
        'cinnamon roll bun',
        'freshly baked bun roll'
    ]
}

# Download images per category with courteous rate-limit delays
for category, queries in category_queries.items():
    dest = path / category
    dest.mkdir(exist_ok=True, parents=True)
    print(f"\n=== Downloading category: '{category}' into '{dest}' ===")
    
    for q in queries:
        urls = search_images(q, max_images=30)
        download_images(dest, urls=urls)
        # Delay between queries to prevent IP throttling
        time.sleep(3)

print(f"\nDownload phase complete. Total files on disk: {len(get_image_files(path))}")


## 4. Image Validation & Data Sanitization

### 1. Practical Goal
Verify all downloaded image binaries on disk and safely delete any truncated, corrupt, or unreadable files.

### 2. Core ML Concept: Data Sanitization & Pipeline Robustness
Web scrapers often encounter interrupted socket streams, zero-byte payloads, HTML error pages saved with `.jpg` extensions, or unsupported compression formats. If unhandled, a single corrupted image will crash the asynchronous PyTorch `DataLoader` worker threads mid-epoch, aborting a long training run. Validating files upfront ensures complete pipeline reliability.

### 3. Python & FastAI Implementation Details
* `get_image_files(path)`: Recursively scans a directory tree and returns all file paths with valid image extensions.
* `verify_images(image_files)`: Concurrently attempts to decode and open each file using PIL. If opening or parsing fails, the path is flagged.
* `failed.map(Path.unlink)`: Functional programming idiom in FastAI (`L.map`) applying `Path.unlink` (deletion) to each corrupted file path.


In [ ]:
# Retrieve all downloaded image paths
image_files = get_image_files(path)
print(f"Inspecting {len(image_files)} downloaded image files for corruption...")

# Detect damaged, unreadable, or truncated image files
failed = verify_images(image_files)

# Unlink (delete) all corrupted files from disk
failed.map(Path.unlink)

valid_files = get_image_files(path)
print(f"Pruned {len(failed)} unreadable/corrupted files.")
print(f"Verified dataset size: {len(valid_files)} valid images.")


## 5. DataBlock Definition & DataLoaders Construction

### 1. Practical Goal
Assemble the declarative `DataBlock` blueprint that translates raw image files into PyTorch-ready training and validation batches.

### 2. Core ML Concept: The DataBlock Architecture & Train/Validation Split
FastAI's `DataBlock` provides a flexible mid-level blueprint separating five core data ingestion questions:
1. *What are the input and target types?* (`ImageBlock` for input pixels, `CategoryBlock` for categorical labels).
2. *Where do the items come from?* (`get_image_files`).
3. *How to split train vs. validation sets?* An 80/20 random split (`RandomSplitter(0.2)`) with a fixed random seed (`seed=42`) ensures benchmark repeatability.
4. *How to extract labels?* The parent folder name (`get_y=parent_label`, assigning `'cat'` or `'bun'`).
5. *How to transform images?* Applying Fastbook Presizing.

### 3. Python & FastAI Implementation Details
* `DataBlock(...)`: The blueprint definition.
* `.dataloaders(path, bs=32)`: Spawns PyTorch `DataLoader` instances with mini-batches of size 32.
* `dls.vocab`: The sorted list of unique category labels derived automatically from the dataset.


In [ ]:
# Define the DataBlock blueprint for Cats and Buns
cats_and_buns_block = DataBlock(
    blocks=(ImageBlock, CategoryBlock),                     # Inputs are images; targets are discrete categories
    get_items=get_image_files,                              # Recursive file discovery
    splitter=RandomSplitter(valid_pct=0.2, seed=42),        # 80% train / 20% validation split (deterministic seed)
    get_y=parent_label,                                     # Label is determined by parent directory name ('cat'/'bun')
    item_tfms=Resize(460),                                  # Presizing Step 1 (CPU): large canvas resize
    batch_tfms=aug_transforms(size=224, min_scale=0.75)     # Presizing Step 2 (GPU): batch data augmentation
)

# Instantiate the DataLoaders with a batch size of 32
dls = cats_and_buns_block.dataloaders(path, bs=32)

print(f"Dataset Vocabulary: {dls.vocab}")
print(f"Training set count:   {len(dls.train_ds)} samples")
print(f"Validation set count: {len(dls.valid_ds)} samples")


## 6. Presizing & Data Augmentation Inspection

### 1. Practical Goal
Visually inspect a training batch to verify that the Presizing and data augmentation pipeline produces diverse, high-quality, non-distorted images.

### 2. Core ML Concept: Fastbook Presizing vs. Naive Augmentation
Naive image augmentation often applies multiple consecutive transformations (cropping, rotating, zooming), each requiring pixel interpolation that compounds blurriness and destroys high-frequency features (such as cat whiskers or bread crust cracks). Furthermore, simple squishing distorts aspect ratios.

**Fastbook Presizing** solves this through a two-stage strategy:
1. **CPU Item Transform (`Resize(460)`)**: Resizes all images to a large common canvas dimension, preserving aspect ratio and preventing empty border margins.
2. **GPU Batch Transform (`aug_transforms(size=224)`)**: Combines random cropping, affine rotations, perspective shifts, and lighting adjustments into a **single GPU tensor operation** with one final interpolation pass.

### 3. Python & FastAI Implementation Details
* `dls.show_batch(max_n=8, nrows=2)`: Visualizes up to 8 augmented images from the training DataLoader with their assigned class labels.


In [ ]:
# Render a batch of augmented training items to inspect data quality
dls.show_batch(max_n=8, nrows=2)


## 7. Transfer Learning & ResNet-34 Learner Setup

### 1. Practical Goal
Initialize a convolutional neural network with pretrained ImageNet weights using the ResNet-34 architecture and attach a new domain-specific classification head.

### 2. Core ML Concept: Transfer Learning & Inductive Bias
Training deep convolutional networks from random scratch weights requires millions of images and days of compute. **Transfer Learning** overcomes this by starting from a model (ResNet-34) pretrained on ImageNet (1.28 million images, 1,000 classes). The early and intermediate convolutional layers have already learned generalized visual features (edges, textures, geometric shapes, animal fur, baked textures).

FastAI strips ImageNet's original 1,000-output linear layer and attaches a modern custom classification head:
- Adaptive Average Pooling & Adaptive Max Pooling (concatenated).
- Batch Normalization & Dropout (regularization against overfitting).
- Linear layer with activations passed through **Softmax** across our 2 target classes.

### 3. Python & FastAI Implementation Details
* `vision_learner(dls, resnet34, metrics=error_rate)`: Builds the Learner object connecting the data loaders, architecture, loss function (`CrossEntropyLossFlat`), and evaluation metric (`error_rate = 1 - accuracy`).


In [ ]:
# Initialize the Vision Learner with ResNet-34 pretrained on ImageNet
learn = vision_learner(
    dls,
    resnet34,
    metrics=error_rate
)

# Print architecture summary details
print("Model Architecture: ResNet-34")
print(f"Default Loss Function: {learn.loss_func}")
print(f"Output vocabulary classes: {learn.dls.vocab}")


## 8. Model Training & Two-Phase Fine-Tuning

### 1. Practical Goal
Fine-tune the neural network using FastAI's automated two-phase transfer learning routine `fine_tune()`.

### 2. Core ML Concept: Discriminative Learning Rates & The 1Cycle Policy
If we immediately trained the entire network with a high learning rate, the large random gradients from the uninitialized new head would destroy the carefully pretrained convolutional weights in the backbone.

`learn.fine_tune(epochs=3, base_lr=2e-3)` implements a robust two-stage strategy:
1. **Phase 1 (Frozen Backbone, 1 Epoch)**: The pretrained ResNet-34 backbone is frozen. Only the newly attached classification head is trained, allowing its weights to reach reasonable values without corrupting the backbone.
2. **Phase 2 (Unfrozen Backbone, 3 Epochs)**: All layers are unfrozen and trained using Leslie Smith's **1Cycle Policy**: the learning rate ramps up and down following a cosine annealing schedule while momentum varies inversely, accelerating convergence while guarding against overfitting.

### 3. Python & FastAI Implementation Details
* `learn.fine_tune(epochs=3, base_lr=2e-3)`: Runs the two-stage transfer learning routine and outputs tabular training loss, validation loss, and error rate per epoch.


In [ ]:
# Train the model using FastAI's two-stage fine-tuning routine
# Stage 1: 1 epoch with frozen backbone
# Stage 2: 3 epochs with unfrozen layers and 1cycle policy
learn.fine_tune(epochs=3, base_lr=2e-3)


## 9. Error Diagnostics: Confusion Matrix

### 1. Practical Goal
Evaluate the model's classification performance across each class on the held-out validation set using a Confusion Matrix.

### 2. Core ML Concept: Beyond Accuracy — Diagnosing Class Bias
A single summary metric like accuracy or error rate can conceal significant operational deficiencies. A **Confusion Matrix** cross-tabulates actual ground-truth classes (rows) against the model's predicted classes (columns):
- **Diagonal cells**: True Positives (correctly classified cats and correctly classified buns).
- **Off-diagonal cells**: Errors. This reveals whether the network exhibits directional bias (e.g., if it tends to classify difficult buns as cats, or vice versa).

### 3. Python & FastAI Implementation Details
* `ClassificationInterpretation.from_learner(learn)`: Computes validation predictions, losses, and confusion statistics.
* `interp.plot_confusion_matrix()`: Generates an annotated heatmap of prediction counts.


In [ ]:
# Compute classification interpretation statistics on the validation set
interp = ClassificationInterpretation.from_learner(learn)

# Render the Confusion Matrix heatmap
interp.plot_confusion_matrix()


## 10. Error Diagnostics: Top Losses Inspection

### 1. Practical Goal
Inspect the specific validation images where the model suffered its highest classification loss.

### 2. Core ML Concept: Hard Negative Mining & Ambiguity Analysis
In cross-entropy loss, high loss occurs when:
1. The model predicts the wrong class with high confidence.
2. The model predicts the correct class, but with very low confidence.

Visualizing top losses allows developers to audit their dataset for:
- **Noisy or incorrect web scraping labels** (e.g., a photo of a dog or a person that was mislabeled as a cat).
- **Legitimate edge cases** (e.g., a golden-brown tabby curled so tightly into an oval that it looks indistinguishable from a brioche bun).

### 3. Python & FastAI Implementation Details
* `interp.plot_top_losses(k=4, nrows=2)`: Visualizes the top $k$ highest-loss images, displaying:
  `prediction / actual / loss / probability of actual class`.


In [ ]:
# Display the top 4 validation images with the highest classification loss
interp.plot_top_losses(k=4, nrows=2)


## 11. Model Serialization & Export

### 1. Practical Goal
Serialize the trained model, learned parameters, vocabulary, and preprocessing transform pipeline into a single portable `.pkl` artifact.

### 2. Core ML Concept: Model Serialization & Production Decoupling
Training code and production inference code have fundamentally different environments and dependencies. In production, we do not require the training dataset, data splitters, loss functions, or optimizers.

FastAI's `learn.export()` bundles:
- Neural network architecture and trained weight tensors.
- The exact inference transform pipeline (ensuring novel test images are resized and normalized identically to training data).
- The class vocabulary index mapping (`['bun', 'cat']`).

### 3. Python & FastAI Implementation Details
* `learn.export(model_path)`: Uses Python's `pickle` under the hood to serialize the learner bundle to disk.
* `model_path.stat().st_size`: Computes file size to verify successful export.


In [ ]:
# Create destination directory for serialized models
model_dir = Path('models')
model_dir.mkdir(exist_ok=True, parents=True)
model_path = model_dir / 'cat_or_bun_model.pkl'

# Export the trained learner bundle
learn.export(model_path)

file_size_mb = model_path.stat().st_size / (1024 * 1024)
print(f"✓ Model successfully serialized to: {model_path}")
print(f"✓ Export bundle size: {file_size_mb:.2f} MB")


## 12. Multi-Class Probability Inference & Visual Display

### 1. Practical Goal
Load the exported model independently and predict the non-binary probability distribution for a novel image, rendering a dual visualization with an image preview and a horizontal probability bar plot.

### 2. Core ML Concept: Softmax Distribution vs. Binary Thresholding
Standard binary classifiers often output a single scalar probability and apply an arbitrary 0.5 threshold (`is_cat: True/False`). However, in real-world scenarios—especially when dealing with visually ambiguous subjects like "Cats vs. Buns"—a single binary label discards crucial nuance.

Instead, the network applies the **Softmax function** across all logits:
$$\sigma(z)_i = \frac{e^{z_i}}{\sum_{j=1}^K e^{z_j}}$$
This guarantees:
1. Every class $i$ receives a valid probability score: $0 \le P(i) \le 1$.
2. The probabilities across all classes sum strictly to 100%: $\sum_i P(i) = 1.0$.

Displaying the full distribution (`{'Cat': 84.0%, 'Bun': 16.0%}`) transparently communicates the model's confidence and ambiguity to end users.

### 3. Python & FastAI Implementation Details
* `load_learner('models/cat_or_bun_model.pkl')`: Restores the exported learner on CPU or GPU without needing training data.
* `learn_inf.predict(img)`: Returns predicted class string, integer class index, and tensor of softmax probabilities.
* `matplotlib.pyplot.subplots(1, 2)`: Displays the input image alongside a horizontal bar chart (`ax.barh`).


In [ ]:
# Load the exported learner independently (independent of training code)
learn_inf = load_learner('models/cat_or_bun_model.pkl')

def predict_cat_or_bun(img_path):
    """
    Predict the non-binary probability distribution for a test image and render
    a side-by-side visualization with a horizontal probability bar plot.
    
    Args:
        img_path (str or Path): Path to the test image file.
    """
    img_path = Path(img_path)
    if not img_path.exists():
        print(f"Error: File not found at {img_path}")
        return
    
    # Load image and compute prediction
    img = PILImage.create(img_path)
    pred_class, pred_idx, probs = learn_inf.predict(img)
    vocab = learn_inf.dls.vocab
    
    # Construct category probability dictionary
    prob_dict = {cat.capitalize(): float(p) for cat, p in zip(vocab, probs)}
    
    print(f"\n==================================================")
    print(f" Inferred Image: {img_path.name}")
    print(f" Top Category:   {pred_class.upper()} ({probs[pred_idx]*100:.1f}%)")
    print(f"--------------------------------------------------")
    print(f" Full Probability Distribution:")
    for cat, prob in prob_dict.items():
        print(f"   • {cat:<6}: {prob*100:5.1f}%")
    print(f"==================================================")
    
    # Side-by-side visualization
    fig, (ax_img, ax_bar) = plt.subplots(1, 2, figsize=(10, 4))
    
    # Left subplot: Display image
    ax_img.imshow(img)
    ax_img.set_title(f"Input: {img_path.name}", fontsize=11, fontweight='bold')
    ax_img.axis('off')
    
    # Right subplot: Horizontal probability bar chart
    categories = list(prob_dict.keys())
    values = [p * 100 for p in prob_dict.values()]
    colors = ['#ff9999' if c == 'Cat' else '#ffcc99' for c in categories]
    
    bars = ax_bar.barh(categories, values, color=colors, edgecolor='#333333', height=0.45)
    ax_bar.set_xlim(0, 100)
    ax_bar.set_xlabel('Model Confidence (%)', fontsize=10)
    ax_bar.set_title('Category Probabilities (Softmax)', fontsize=11, fontweight='bold')
    ax_bar.grid(axis='x', linestyle='--', alpha=0.5)
    
    # Annotate bar percentages
    for bar in bars:
        w = bar.get_width()
        ax_bar.text(w + 2, bar.get_y() + bar.get_height()/2, f'{w:.1f}%',
                    va='center', ha='left', fontsize=10, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

# Run inference test on the first available verified image
verified_samples = get_image_files(path)
if len(verified_samples) > 0:
    sample_to_test = verified_samples[0]
    predict_cat_or_bun(sample_to_test)
else:
    print("Dataset directory is empty. Run image download steps (Cells 2-4) first.")
